# SLR SLR: train the GrocerEye-style detector on Colab (TR-3)

Stage A: reproduce GrocerEye (YOLOv3, 84.59% mAP) with YOLO11 nano on the 5K Groceries / Freiburg
dataset, then evaluate on the held-out test split and download `best.pt` + the eval report.

1. **Runtime -> Change runtime type -> GPU** (T4 is fine).
2. Run the cells top to bottom. 100 epochs at imgsz 320 takes roughly 30-60 min on a T4.

## 1. Get the code

The repo `Ore-Porat/VisuallyImpairedBodycam` is **private**, so a plain `git clone` fails. Either:

- **Token:** create a GitHub fine-grained personal access token with read access to the repo
  (GitHub -> Settings -> Developer settings), then run the next cell and paste it when asked
  (it is not saved in the notebook), or
- **Zip upload:** on your laptop zip the repo (without `.venv/`, `data/`, `runs/`), upload it with
  the Files panel on the left, set `USE_ZIP = True` below and put the file name in `ZIP_NAME`.

In [ ]:
import os, getpass, subprocess
USE_ZIP = False
ZIP_NAME = "VisuallyImpairedBodycam.zip"
REPO = "Ore-Porat/VisuallyImpairedBodycam"

%cd /content
if USE_ZIP:
    !unzip -q -o {ZIP_NAME} -d /content/repo_zip
    # the zip may contain a top-level folder; find the one with training/
    root = next(r for r, d, f in os.walk("/content/repo_zip") if "training" in d)
    !rm -rf /content/VisuallyImpairedBodycam && cp -r "{root}" /content/VisuallyImpairedBodycam
elif not os.path.isdir("/content/VisuallyImpairedBodycam"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1",
                    f"https://{token}@github.com/{REPO}.git", "/content/VisuallyImpairedBodycam"], check=True)
    del token
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", "/content/VisuallyImpairedBodycam", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
%cd /content/VisuallyImpairedBodycam

## 2. Install

In [ ]:
!pip install -q "ultralytics>=8.3" albumentations pyyaml
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Download + convert the dataset (TR-1, TR-2)

Downloads the 5K Groceries annotations (~375 MB) from
github.com/aleksandar-aleksandrov/groceries-object-detection-dataset, converts VOC XML to YOLO and
makes the stratified 80/10/10 split (seed 0) in `data/grocereye/`.

In [ ]:
!python training/get_grocereye_data.py

## 4. Train (Stage A)

Set `NAME` / `EPOCHS` as needed. For a quick check first, add `--epochs 1 --fraction 0.05`.

In [ ]:
NAME = "grocereye_a"
EPOCHS = 100
!python training/train.py --name {NAME} --epochs {EPOCHS} --imgsz 320 --batch 64 --device 0 --workers 2

## 5. Evaluate on the test split (TR-4, NFR-5)

In [ ]:
!python training/evaluate.py --weights models/{NAME}/best.pt --split test --device 0
from IPython.display import Markdown, display
display(Markdown(open(f"models/{NAME}/eval_report.md").read()))

## 6. Download the weights and report

Put `best.pt` in `models/<NAME>/` on the laptop and commit the eval report (weights are gitignored).
Then set `model.weights: models/<NAME>/best.pt` and `model.classes_file: config/classes.yaml` in
`config/server.yaml`.

In [ ]:
!cd models && zip -q -r /content/{NAME}.zip {NAME}
!cp reports/eval_{NAME}_*.md /content/ 2>/dev/null || true
from google.colab import files
files.download(f"/content/{NAME}.zip")